In [ ]:
import os

from dotenv import load_dotenv
from azure.cosmos import CosmosClient
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient

loaded = load_dotenv()
if not loaded:
    raise RuntimeError(".env file could not be loaded.")

key_vault_url = os.environ["KEY_VAULT_URL"]
secret_name = os.environ["COSMOS_DB_CONNECTION_STRING_SECRET_NAME"]
credential = DefaultAzureCredential()
secret_client = SecretClient(vault_url=key_vault_url, credential=credential)
connection_string = secret_client.get_secret(secret_name).value

if connection_string is None:
    raise RuntimeError(f"Key Vault secret '{secret_name}' has no value.")

client = CosmosClient.from_connection_string(connection_string)

DatabaseProxy = client.create_database_if_not_exists("test_db_5");

ContainerProxy = DatabaseProxy.create_container_if_not_exists(
  id="product",
  partition_key={"paths": ["/categoryId"], "kind": "Hash"}                                    
)

# Create a document
ContainerProxy.upsert_item({"id": "1", "categoryId": "electronics"})

items = ContainerProxy.query_items(query="SELECT * FROM product", enable_cross_partition_query=True)

{'id': '1',
 'categoryId': 'electronics',
 '_rid': 'tq8qANY0TQYBAAAAAAAAAA==',
 '_self': 'dbs/tq8qAA==/colls/tq8qANY0TQY=/docs/tq8qANY0TQYBAAAAAAAAAA==/',
 '_etag': '"7001d09f-0000-0200-0000-6a9932050000"',
 '_attachments': 'attachments/',
 '_ts': 1788424709}